# Azure 音频 Whisper（预览版）示例

本示例展示如何使用 Azure OpenAI Whisper 模型转录音频文件。


## 环境准备

首先，我们安装必要的依赖并导入要用到的库。

In [ ]:
! pip install "openai>=1.0.0,<2.0.0"
! pip install python-dotenv

In [ ]:
import os
import openai
import dotenv

dotenv.load_dotenv()

### 身份验证

Azure OpenAI 服务支持多种身份验证机制，包括 API 密钥和 Azure Active Directory 令牌凭据。

In [4]:
use_azure_active_directory = False  # Set this flag to True if you are using Azure Active Directory

#### 使用 API 密钥进行身份验证

要将 OpenAI SDK 配置为使用 *Azure API 密钥*，需要把 `api_key` 设置为与你的终结点关联的密钥（你可以在 [Azure 门户](https://portal.azure.com)的 *"资源管理"* 下的 *"密钥和终结点"* 中找到该密钥）。你也会在这里找到资源的终结点。

In [5]:
if not use_azure_active_directory:
    endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
    api_key = os.environ["AZURE_OPENAI_API_KEY"]

    client = openai.AzureOpenAI(
        azure_endpoint=endpoint,
        api_key=api_key,
        api_version="2023-09-01-preview"
    )

#### 使用 Azure Active Directory 进行身份验证
现在来看看如何通过 Azure Active Directory 进行身份验证。我们先安装 `azure-identity` 库。该库会提供身份验证所需的令牌凭据，并通过 `get_bearer_token_provider` 辅助函数帮助我们构建令牌凭据提供程序。建议使用 `get_bearer_token_provider`，而不是向 `AzureOpenAI` 提供静态令牌，因为该 API 会自动为你缓存和刷新令牌。

有关如何为 Azure OpenAI 设置 Azure Active Directory 身份验证的更多信息，请参阅[文档](https://learn.microsoft.com/azure/ai-services/openai/how-to/managed-identity)。

In [ ]:
! pip install "azure-identity>=1.15.0"

In [5]:
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

if use_azure_active_directory:
    endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
    api_key = os.environ["AZURE_OPENAI_API_KEY"]

    client = openai.AzureOpenAI(
        azure_endpoint=endpoint,
        azure_ad_token_provider=get_bearer_token_provider(DefaultAzureCredential(), "https://cognitiveservices.azure.com/.default"),
        api_version="2023-09-01-preview"
    )

> 注意：如果未提供以下参数，AzureOpenAI 会从对应的环境变量中推断它们：

- `api_key` 来自 `AZURE_OPENAI_API_KEY`
- `azure_ad_token` 来自 `AZURE_OPENAI_AD_TOKEN`
- `api_version` 来自 `OPENAI_API_VERSION`
- `azure_endpoint` 来自 `AZURE_OPENAI_ENDPOINT`


## 部署

在本节中，我们将使用 `whisper-1` 模型创建一个部署，用于转录音频文件。

### 部署：在 Azure OpenAI Studio 中创建
我们来部署一个用于 Whisper 的模型。访问 https://portal.azure.com, 找到你的 Azure OpenAI 资源，然后进入 Azure OpenAI Studio。点击 "Deployments" 选项卡，为你想用于 Whisper 的模型创建一个部署。你为模型指定的部署名称将在下面的代码中使用。

In [6]:
deployment = "whisper-deployment" # Fill in the deployment name from the portal here

## 音频转录

音频转录（即语音转文本）是将口语转换为文字的过程。使用 `openai.Audio.transcribe` 方法可以将音频文件流转录为文本。

你可以从 [GitHub 上的 Azure AI Speech SDK 仓库](https://github.com/Azure-Samples/cognitive-services-speech-sdk/tree/master/sampledata/audiofiles)获取示例音频文件。

In [7]:
# download sample audio file
import requests

sample_audio_url = "https://github.com/Azure-Samples/cognitive-services-speech-sdk/raw/master/sampledata/audiofiles/wikipediaOcelot.wav"
audio_file = requests.get(sample_audio_url)
with open("wikipediaOcelot.wav", "wb") as f:
    f.write(audio_file.content)

In [ ]:
transcription = client.audio.transcriptions.create(
    file=open("wikipediaOcelot.wav", "rb"),
    model=deployment,
)
print(transcription.text)